In [14]:
import pandas as pd
import numpy as np
import torch
import collections
from pathlib import Path

DATA_PATH = "../EHRSHOT/EHRSHOT_MEDS/data/data.parquet"
VOCAB_PATH = "vocab.txt"
BINS = 5  # number of quantile bins for numeric values

In [15]:
df = pd.read_parquet(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Patients: {df['subject_id'].nunique()}")
print(df["code"].str.split("/").str[0].value_counts())

Shape: (41661637, 7)
Patients: 6739
code
LOINC                   32095577
SNOMED                   4769737
RxNorm                   2349784
CPT4                     1162788
Visit                     380317
Domain                    316295
CARE_SITE                 199469
Medicare Specialty        178767
RxNorm Extension           99961
CMS Place of Service       47714
ICD10PCS                   19759
ICD9Proc                    8468
Cancer Modifier             7750
Gender                      6739
Ethnicity                   6274
Race                        5176
CVX                         3483
HCPCS                       1212
OMOP Extension               935
ICDO3                        767
Condition Type               665
Name: count, dtype: int64


In [16]:
def bin_numeric_codes(df, n_bins=BINS):
    df = df.copy()
    df["code_token"] = df["code"]  # default: no binning

    has_value = df["numeric_value"].notna() & (df["numeric_value"] != 0)

    for code, group in df[has_value].groupby("code"):
        values = group["numeric_value"].dropna()
        # Skip if not enough unique values to form bins
        if values.nunique() < n_bins:
            continue
        try:
            binned = pd.qcut(values, q=n_bins, labels=False, duplicates="drop")
            bin_labels = binned.map(lambda b: f"{code}_B{int(b)}" if pd.notna(b) else code)
            df.loc[bin_labels.index, "code_token"] = bin_labels
        except Exception:
            continue  # fall back to raw code if binning fails for any reason

    return df

df = bin_numeric_codes(df)
print("Sample binned codes:")
print(df[df["code_token"].str.contains("_B")]["code_token"].value_counts().head(10))

/Users/adrienbelanger/.pyenv/versions/3.12.2/lib/python3.12/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


Sample binned codes:
code_token
LOINC/8867-4_B0    556237
LOINC/8867-4_B1    536279
LOINC/8867-4_B3    518356
LOINC/8867-4_B4    514352
LOINC/8867-4_B2    485671
LOINC/8480-6_B1    401199
LOINC/8480-6_B3    397322
LOINC/8480-6_B0    390135
LOINC/8480-6_B2    380483
LOINC/8480-6_B4    376453
Name: count, dtype: int64


In [17]:
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

all_codes = df["code_token"].dropna().unique().tolist()
vocab = collections.OrderedDict()
for i, tok in enumerate(SPECIAL_TOKENS + sorted(all_codes)):
    vocab[tok] = i

Path(VOCAB_PATH).parent.mkdir(exist_ok=True)
with open(VOCAB_PATH, "w") as f:
    for token in vocab:
        f.write(token + "\n")

print(f"Vocab size: {len(vocab)}")
print(f"Special tokens: {SPECIAL_TOKENS}")
print(f"Sample codes: {list(vocab.keys())[5:10]}")

Vocab size: 37256
Special tokens: ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']
Sample codes: ['CARE_SITE/7928118', 'CARE_SITE/7928122', 'CARE_SITE/7928123', 'CARE_SITE/7928167', 'CARE_SITE/7928169']


In [18]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class MEDSDataset(Dataset):
    def __init__(self, df, vocab, max_length=256):
        self.max_length = max_length
        self.vocab = vocab
        self.pad_idx = vocab["[PAD]"]
        self.unk_idx = vocab["[UNK]"]
        self.data = self._preprocess(df)

    def _preprocess(self, df):
        df = df.sort_values(["subject_id", "time"])
        grouped = df.groupby("subject_id").agg(
            code_tokens=("code_token", list),
            timestamps=("time", list)
        ).reset_index()
        return grouped

    def _tokenize(self, codes):
        return [self.vocab.get(c, self.unk_idx) for c in codes]

    def _to_time_deltas(self, timestamps):
        """Convert timestamps to time deltas in days from first event"""
        ts = pd.to_datetime(timestamps)
        deltas = [(t - ts[0]).days for t in ts]
        return deltas

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        patient = self.data.iloc[idx]

        tokens = self._tokenize(patient.code_tokens)
        deltas = self._to_time_deltas(patient.timestamps)

        # Truncate to last max_length events
        if len(tokens) > self.max_length:
            tokens = tokens[-self.max_length:]
            deltas = deltas[-self.max_length:]

        # Pad
        pad_len = self.max_length - len(tokens)
        tokens  = tokens  + [self.pad_idx] * pad_len
        deltas  = deltas  + [deltas[-1]]   * pad_len  # pad with last delta

        return (
            torch.tensor(tokens, dtype=torch.long),
            torch.tensor(deltas, dtype=torch.float32)
        )


def make_dataloaders(df, vocab, batch_size=32, max_length=256):
    patient_ids = df["subject_id"].unique()
    train_ids, temp_ids   = train_test_split(patient_ids, test_size=0.2, random_state=42)
    valid_ids, test_ids   = train_test_split(temp_ids,    test_size=0.5, random_state=42)

    train_df = df[df["subject_id"].isin(train_ids)]
    valid_df = df[df["subject_id"].isin(valid_ids)]
    test_df  = df[df["subject_id"].isin(test_ids)]

    train_ds = MEDSDataset(train_df, vocab, max_length)
    valid_ds = MEDSDataset(valid_df, vocab, max_length)
    test_ds  = MEDSDataset(test_df,  vocab, max_length)

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True),
        DataLoader(valid_ds, batch_size=batch_size, shuffle=False),
        DataLoader(test_ds,  batch_size=batch_size, shuffle=False),
    )

In [19]:
train_loader, valid_loader, test_loader = make_dataloaders(df, vocab)

tokens, deltas = next(iter(train_loader))
print("Token batch shape:", tokens.shape)   # expect [32, 256]
print("Delta batch shape:", deltas.shape)   # expect [32, 256]
print("Sample tokens:", tokens[0, :10])
print("Sample deltas:", deltas[0, :10])
print(f"Train patients: {len(train_loader.dataset)}, "
      f"Valid: {len(valid_loader.dataset)}, "
      f"Test:  {len(test_loader.dataset)}")

Token batch shape: torch.Size([32, 256])
Delta batch shape: torch.Size([32, 256])
Sample tokens: tensor([17939, 17939, 13396, 14324, 24526, 27616, 27618, 27640, 27642, 27641])
Sample deltas: tensor([24532., 24532., 24532., 24532., 24532., 24532., 24532., 24532., 24532.,
        24532.])
Train patients: 5391, Valid: 674, Test:  674


In [26]:
from layers.configs import TrajGPTConfig

configs = TrajGPTConfig(
    num_layers=2,
    num_heads=2,
    d_model=64,
    qk_dim=64,
    v_dim=128,
    ffn_proj_size=256,
)
configs.vocab_size   = len(vocab)
configs.dropout      = 0.1
configs.use_grad_ckp = False

print(f"vocab_size:   {configs.vocab_size}")
print(f"d_model:      {configs.d_model}")
print(f"num_layers:   {configs.num_layers}")
print(f"num_heads:    {configs.num_heads}")

vocab_size:   37256
d_model:      64
num_layers:   2
num_heads:    2


In [27]:
from models.TrajGPT_EHR import TrajGPT

model = TrajGPT(configs, head_type='pretrain')
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

tokens, deltas = next(iter(train_loader))
loss = model(tokens, deltas, y=tokens)
print(f"Forward pass loss: {loss.item():.4f}")

Parameters: 4,938,769
Forward pass loss: 10.7131


In [ ]:
import os
import time
import numpy as np
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR

# --- Config ---
TRAIN_EPOCHS  = 20
BATCH_SIZE    = 32   # keep lower for MPS stability
LR            = 3e-5
ACCUM_STEPS   = 4    # effective batch size = 32 * 4 = 128, matching original
PATIENCE      = 3
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- Device ---
device = (
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("cpu")
)
print(f"Using device: {device}")

# --- Rebuild dataloaders with correct batch size ---
train_loader, valid_loader, test_loader = make_dataloaders(
    df, vocab, batch_size=BATCH_SIZE, max_length=256
)

# --- Model ---
model = TrajGPT(configs, head_type='pretrain').to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = OneCycleLR(
    optimizer,
    max_lr=LR,
    steps_per_epoch=len(train_loader) // ACCUM_STEPS,
    epochs=TRAIN_EPOCHS,
    pct_start=0.3
)
# Cell — diagnose device performance
import time

print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"Model device: {next(model.parameters()).device}")

# Time a single batch properly
model.eval()
batch_x, batch_t = next(iter(train_loader))
batch_x, batch_t = batch_x.to(device), batch_t.to(device)

with torch.no_grad():
    t0 = time.time()
    loss = model(X=batch_x, input_time=batch_t, y=batch_x, forward_impl='parallel')
    t1 = time.time()

print(f"Single batch inference time: {t1-t0:.2f}s")
print(f"Batch shape: {batch_x.shape}")

Using device: mps
MPS available: True
Model device: mps:0
Single batch inference time: 72.08s
Batch shape: torch.Size([32, 256])


In [29]:
import os
import time
import numpy as np
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR

# --- Config ---
TRAIN_EPOCHS   = 3
BATCH_SIZE     = 32
LR             = 3e-5
ACCUM_STEPS    = 4
PATIENCE       = 3
CHECKPOINT_DIR = "checkpoints"
N_PATIENTS     = 100  # small subset just to validate loop
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- Subset for smoke test ---
subset_ids = df["subject_id"].unique()[:N_PATIENTS]
df_subset  = df[df["subject_id"].isin(subset_ids)]
train_loader_small, valid_loader_small, _ = make_dataloaders(
    df_subset, vocab, batch_size=BATCH_SIZE, max_length=256
)
print(f"Train batches: {len(train_loader_small)}, Valid batches: {len(valid_loader_small)}")

# --- Model ---
model_tiny = TrajGPT(configs, head_type='pretrain').to(device)
print(f"Parameters: {sum(p.numel() for p in model_tiny.parameters()):,}")

optimizer = optim.Adam(model_tiny.parameters(), lr=LR)
scheduler = OneCycleLR(
    optimizer,
    max_lr=LR,
    steps_per_epoch=max(1, len(train_loader_small) // ACCUM_STEPS),
    epochs=TRAIN_EPOCHS,
    pct_start=0.3
)

# --- Loop ---
best_valid_loss = float("inf")
patience_counter = 0

for epoch in range(TRAIN_EPOCHS):
    model_tiny.train()
    train_losses = []
    optimizer.zero_grad()
    t0 = time.time()

    for i, (batch_x, batch_t) in enumerate(train_loader_small):
        batch_x = batch_x.to(device)
        batch_t = batch_t.to(device)

        loss = model_tiny(X=batch_x, input_time=batch_t, y=batch_x, forward_impl='parallel')
        (loss / ACCUM_STEPS).backward()
        train_losses.append(loss.item())

        if (i + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model_tiny.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    model_tiny.eval()
    valid_losses = []
    with torch.no_grad():
        for batch_x, batch_t in valid_loader_small:
            batch_x = batch_x.to(device)
            batch_t = batch_t.to(device)
            loss = model_tiny(X=batch_x, input_time=batch_t, y=batch_x, forward_impl='parallel')
            valid_losses.append(loss.item())

    elapsed = time.time() - t0
    print(f"Epoch {epoch+1:02d}/{TRAIN_EPOCHS} | "
          f"train: {np.mean(train_losses):.4f} | "
          f"valid: {np.mean(valid_losses):.4f} | "
          f"time: {elapsed:.1f}s")

    if np.mean(valid_losses) < best_valid_loss:
        best_valid_loss = np.mean(valid_losses)
        patience_counter = 0
        torch.save(model_tiny.state_dict(), f"{CHECKPOINT_DIR}/best_model_tiny.pt")
        print(f"  ✓ Saved (valid_loss={best_valid_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

print(f"\nDone. Best valid loss: {best_valid_loss:.4f}")

Train batches: 3, Valid batches: 1
Parameters: 4,938,769
Epoch 01/3 | train: 10.6860 | valid: 10.6870 | time: 340.2s
  ✓ Saved (valid_loss=10.6870)
Epoch 02/3 | train: 10.6841 | valid: 10.6870 | time: 314.4s
Epoch 03/3 | train: 10.6855 | valid: 10.6870 | time: 328.5s

Done. Best valid loss: 10.6870


In [ ]:
import pandas as pd
labels = pd.read_csv("EHRSHOT/EHRSHOT_ASSETS/benchmark/all_labels.csv")
print(labels.shape)
print(labels.columns.tolist())
print(labels.head())
print(labels["task"].value_counts())